<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/00-unbounded-table-model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 14.0 — The unbounded-table model, made concrete

Build a streaming DataFrame from the `rate` source, confirm
`isStreaming`, run the SAME query shape you'd write for static data,
and watch the in-memory sink's row count grow across two checks --
the unbounded-table model, observed rather than just described.

In [ ]:
import os, sys, shutil, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.0-unbounded-table-model")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Same DataFrame class, one new flag

In [ ]:
static_df = spark.range(10)
stream_df = spark.readStream.format("rate").option("rowsPerSecond", 3).load()

print("static_df.isStreaming  ->", static_df.isStreaming)
print("stream_df.isStreaming  ->", stream_df.isStreaming)
print("same Python type?      ->", type(static_df) == type(stream_df))

## The exact same query shape, run incrementally

`select`/`withColumn` -- ordinary Module 3 code. The only new thing is
`.writeStream` instead of an action, and a `memory` sink so we can
inspect results with plain SQL as the table grows.

In [ ]:
transformed = stream_df.withColumn("bucket", (col("value") % 5))

query = (transformed.writeStream
         .format("memory")
         .queryName("unbounded_demo")
         .outputMode("append")
         .start())

print("query.isActive ->", query.isActive)

## Watch the "table" grow -- two checks, same query, more rows

This IS the unbounded-table model: the in-memory table only ever gets
appended to, and each check below queries whatever has arrived so far.

In [ ]:
time.sleep(3)
print("after ~3s:")
spark.sql("SELECT count(*) AS n FROM unbounded_demo").show()

time.sleep(3)
print("after ~6s (same query, more rows -- nothing re-run, just appended):")
spark.sql("SELECT count(*) AS n FROM unbounded_demo").show()
spark.sql("SELECT * FROM unbounded_demo ORDER BY value DESC LIMIT 5").show()

In [ ]:
query.stop()
print("query.isActive after stop ->", query.isActive)

## An operation that's NOT valid on the unbounded table

`orderBy` with no watermark over the full stream -- disallowed, because
sorting the whole unbounded table isn't incremental.

In [ ]:
from pyspark.sql.utils import StreamingQueryException, AnalysisException

try:
    bad_query = (stream_df.orderBy("value")   # full sort, no watermark/window
                 .writeStream.format("memory").queryName("bad").outputMode("complete").start())
    bad_query.awaitTermination(3)
except AnalysisException as e:
    print("rejected at query-definition time, exactly as 14.0 predicts:")
    print(str(e).splitlines()[0])

In [ ]:
spark.stop()

## Exercises

1. Build a streaming DataFrame from `rate`, apply `.filter(col("value")
   % 2 == 0)`, run it to a memory sink, and confirm every row in the
   resulting table is even -- ordinary Module 3 filter logic, running
   incrementally.
2. Call `query.status` and `query.recentProgress` on a running query.
   What information do they give you that a static DataFrame's
   `.explain()` doesn't?
3. Try `stream_df.limit(5)` into a memory sink with `outputMode=
   "append"`. Does it behave the way you'd expect from static
   `.limit()`? What does this suggest about which static operations
   translate cleanly to streaming and which don't?
4. Start two streaming queries from the SAME source DataFrame
   (`stream_df`) with two different memory sink names. Do they share
   state, or does each maintain its own independent unbounded table?
5. Trigger the `orderBy` rejection again, but this time with a
   `.groupBy(F.window(...))` first (you'll need an event-time column --
   use `"timestamp"`, the rate source's own column). Does adding a
   window change whether an unbounded sort is legal downstream? Why or
   why not (preview of 14.3).

In [ ]:
# your exercise code here